<img src = "/Users/ashishmeshram/Desktop/Master-Project/My-Implementation/images.png">

In [39]:
import numpy as np

A = np.array([[0,1,0,1,0,0]
            ,[1,0,1,0,0,0],
              [0,1,0,0,1,0],
              [1,0,0,0,1,0],
              [0,0,1,1,0,1],
              [0,0,0,0,1,0]])




In [40]:

du = A.sum(axis=1)

print(du)

[2 2 2 2 3 1]


In [41]:
for i in range(len(du)) :
    print(f"Degree of {i} : {du[i]} " )


P = A / du[:, None]
print(P)

Degree of 0 : 2 
Degree of 1 : 2 
Degree of 2 : 2 
Degree of 3 : 2 
Degree of 4 : 3 
Degree of 5 : 1 
[[0.         0.5        0.         0.5        0.         0.        ]
 [0.5        0.         0.5        0.         0.         0.        ]
 [0.         0.5        0.         0.         0.5        0.        ]
 [0.5        0.         0.         0.         0.5        0.        ]
 [0.         0.         0.33333333 0.33333333 0.         0.33333333]
 [0.         0.         0.         0.         1.         0.        ]]


In [42]:
for i in range (10) : 
    current = 4
    x = np.random.choice(6 , p = P[current])
    print(x)

2
3
5
3
2
5
2
3
5
3


In [43]:
def random_walk(walk_length , start_node, P , du) :
    walk = [start_node]
    current = start_node
    for i in range(walk_length - 1 ) :
        next_node = np.random.choice(len(du) , p=P[current])
        walk.append(next_node)
        current = next_node
    return walk

In [44]:
walk_seq = random_walk(10 , 4, P , du)

In [45]:
walk_seq

[4, 3, 4, 3, 4, 5, 4, 5, 4, 2]

In [46]:
def generate_context_pairs(walk__seq , window_size) :
    pairs = [ ]
    for i , center in enumerate(walk__seq):
        left = max(0 , i - window_size)
        right = min(len(walk__seq) , i + window_size +1)
        
        for j in range(left , right):
            if i ==j :
                continue
            else:
                contex = walk__seq[j]
                pairs.append((center , contex))
    return pairs
    

In [47]:
generate_context_pairs(walk_seq , 2)

[(4, 3),
 (4, 4),
 (3, 4),
 (3, 4),
 (3, 3),
 (4, 4),
 (4, 3),
 (4, 3),
 (4, 4),
 (3, 3),
 (3, 4),
 (3, 4),
 (3, 5),
 (4, 4),
 (4, 3),
 (4, 5),
 (4, 4),
 (5, 3),
 (5, 4),
 (5, 4),
 (5, 5),
 (4, 4),
 (4, 5),
 (4, 5),
 (4, 4),
 (5, 5),
 (5, 4),
 (5, 4),
 (5, 2),
 (4, 4),
 (4, 5),
 (4, 2),
 (2, 5),
 (2, 4)]

# SKIP -GRAM

In [48]:
v_center = [1 , -2 , 3]
v_context = [-1 , 0.6 , -0.7]

score = np.dot(v_center,v_context)
prob = 1/(1+np.exp(-score))

loss = -np.log(prob)

print(f"score : {score}")
print(f'prob : {prob}')
print(f"loss : {loss}")

score : -4.3
prob : 0.013386917827664779
loss : 4.313477330416026


In [49]:
loss_gradient = prob - 1

### Example with Negative Sample

In [50]:
v1 = np.array([0.5 , 0.5])
v2 = np.array([ 0.5, 0.5])
v5 = np.array([0.5, -0.5])

# (v1, v2) + , (v1 , v5) -


positive_score = np.dot(v1 , v2)
negative_score = np.dot(v1, v5)
print(f"score : {positive_score} , {negative_score}")
positive_prob = 1/(1+np.exp(-positive_score))
negative_prob = 1 / (1+np.exp(negative_score))

positive_loss = -np.log(positive_prob)
negative_loss = -np.log(negative_prob)

loss = positive_loss + negative_loss


#GRADIENT

grad_v1 = (positive_prob -1)*v2 + (negative_prob)*v5
grad_v2 = (positive_prob-1)*v1
grad_v5 = negative_prob*v1

lr = 0.5

v1-=lr*grad_v1
v2-=lr*grad_v2
v5-=lr*grad_v5

score : 0.5 , 0.0


In [51]:
print(f"v1 : {v1}")
print(f"v2 : {v2}")
print(f"v5 : {v5}")
print(f"updated score: {np.dot(v1,v2)} , {np.dot(v1,v5)}")

v1 : [0.46938517 0.71938517]
v2 : [0.59438517 0.59438517]
v5 : [ 0.375 -0.625]
updated score: 0.7065874539736416 , -0.2735962917998841


## With pytorch

In [52]:
import torch

In [53]:
v1 = torch.tensor([0.5, 0.5] , dtype =torch.float32 , requires_grad=True)
v2 = torch.tensor([0.5, 0.5] , dtype =torch.float32 , requires_grad=True)
v5 = torch.tensor([0.5, -0.5] , dtype =torch.float32 , requires_grad=True)

In [54]:
positive_score = torch.dot(v1,v2)
negative_score = torch.dot(v1,v5)

loss = -torch.log(torch.sigmoid(positive_score)) - torch.log(torch.sigmoid(-negative_score))

loss.backward()

In [55]:
print(v1.grad , v2.grad , v5.grad)

tensor([ 0.0612, -0.4388]) tensor([-0.1888, -0.1888]) tensor([0.2500, 0.2500])


### EXAMPLE = 6X4

In [56]:
import torch.nn as nn

center_matrix = nn.Embedding(6,4)
context_matrix = nn.Embedding(6,4)

print(center_matrix.weight)
v_center = center_matrix(torch.tensor(1))
v_positive = context_matrix(torch.tensor(2))
v_negative = context_matrix(torch.tensor(5))

positive_score = torch.dot(v_center , v_positive)
negative_score = torch.dot(v_center, v_negative)

positive_loss = -torch.log(torch.sigmoid(positive_score))
negative_loss = -torch.log(torch.sigmoid(-negative_score))

loss = positive_loss+negative_loss

loss.backward()

Parameter containing:
tensor([[ 3.6834e-01, -6.1586e-01, -3.5048e-01,  1.5108e+00],
        [ 1.3040e+00,  5.4500e-01, -2.6348e-01,  3.8222e-01],
        [ 1.3004e+00,  7.8966e-01, -1.0370e-01, -6.4840e-02],
        [ 1.3049e+00,  1.0524e-01, -1.2842e+00,  8.6603e-01],
        [ 6.2336e-01,  3.7618e-01,  1.2487e+00, -9.3586e-04],
        [ 1.9978e-01, -3.7931e-01, -3.3596e-02, -9.0639e-01]],
       requires_grad=True)


In [57]:
print(center_matrix.weight)

Parameter containing:
tensor([[ 3.6834e-01, -6.1586e-01, -3.5048e-01,  1.5108e+00],
        [ 1.3040e+00,  5.4500e-01, -2.6348e-01,  3.8222e-01],
        [ 1.3004e+00,  7.8966e-01, -1.0370e-01, -6.4840e-02],
        [ 1.3049e+00,  1.0524e-01, -1.2842e+00,  8.6603e-01],
        [ 6.2336e-01,  3.7618e-01,  1.2487e+00, -9.3586e-04],
        [ 1.9978e-01, -3.7931e-01, -3.3596e-02, -9.0639e-01]],
       requires_grad=True)


In [58]:
type(center_matrix)

torch.nn.modules.sparse.Embedding

In [59]:
(center_matrix(torch.tensor(1)))

tensor([ 1.3040,  0.5450, -0.2635,  0.3822], grad_fn=<EmbeddingBackward0>)

### WITH BATCH

In [66]:
import torch.nn as nn
num_nodes = 6
num_dim = 4 

center_matrix = nn.Embedding(num_nodes , num_dim)
context_matrix =  nn.Embedding(num_nodes , num_dim)


center = torch.tensor([1, 4, 2,  1])
contex = torch.tensor([2,3,1,0])
negative = torch.tensor([5,0,5,4])


v_center = center_matrix(center)
v_positive = context_matrix(contex)
v_negative = context_matrix(negative)

positive_score = (v_center * v_positive).sum(dim =1)
negative_score = (v_center * v_negative).sum(dim =1)

loss = (-torch.log(torch.sigmoid(positive_score)) - torch.log(torch.sigmoid(-negative_score))).mean()

loss.backward()

In [67]:
print(f"batch loss : {loss}")

batch loss : 1.907565712928772


In [72]:


positive_score.shape

torch.Size([4])